# Hardware reality check: Grover's search on three different QPUs

*Part of the QUEST Foundations & Algorithms series*

Most quantum computing courses teach Grover's algorithm as if the hardware is invisible. You build the circuit, run it on a simulator, get the marked item back with high probability, done. Run the same circuit on real hardware and the results differ between vendors, sometimes by a lot. I'd argue that understanding why teaches you more than a hundred additional simulator runs.

This notebook takes a single 4-qubit Grover circuit and submits it to three quantum processors: a superconducting device from Rigetti, a superconducting device from IQM with a different qubit topology, and a trapped-ion device from AQT. We compare how each performs, look at what the compilers produced for each backend, and use those differences to build intuition about designing quantum algorithms for real hardware.

**Learning objectives.** By the end of this notebook you will:

1. Build and understand a standard Grover's search circuit for 4 qubits.
2. Submit the same logical circuit to three physically different backends via qBraid's unified API.
3. Compare measured success probabilities against the theoretical curve.
4. Read the transpiled output for each backend and identify how native gate sets and connectivity change the circuit.
5. Reason about what hardware differences mean for algorithm design.

**What to bring in.** You should have seen Grover's algorithm at least once in class. If not, the review section below is enough to follow along, but the deeper points will land better after your lecture.

**Credit budget.** This notebook consumes approximately 1,800 shots per full run (150 shots × 3 backends × 4 iteration counts). At a typical cost of a few cents per hundred shots, one execution costs on the order of a dollar in credits.


## Grover's algorithm, the two-sentence version

Grover's algorithm finds a marked item in an unstructured database of $N$ items using approximately $\frac{\pi}{4}\sqrt{N}$ queries to an oracle, compared to $N/2$ on average classically. The trick is that we don't measure between queries. Instead we alternate an oracle (which flips the phase of the marked state) with a diffusion operator (which reflects amplitudes around their mean).

For $n$ qubits, $N = 2^n$. The success probability after $k$ Grover iterations is:

$$P_{\text{success}}(k) = \sin^2\left((2k + 1)\theta\right), \quad \text{where } \sin(\theta) = \frac{1}{\sqrt{N}}$$

For $n=4$ qubits ($N=16$), the optimal number of iterations is $\lfloor \frac{\pi}{4}\sqrt{16} \rfloor = 3$, at which point the success probability peaks near $96\%$ on an ideal quantum computer.

*The success probability is a smooth function of $k$. Run $k$ from 0 to 5 and you should see it rise, peak, and fall back down. That fall is the part worth watching for. It is the signature of Grover's algorithm working correctly.*


## Setup

In [ ]:
# Standard scientific Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110
np.random.seed(42)  # reproducible randomization

print("Setup complete.")

## Building the circuit

We're going to mark the state $|1010\rangle$ (binary for the decimal number 10). The oracle will flip the phase of this state and leave all others untouched. The diffusion operator will then reflect amplitudes around the mean.

We build the oracle using a multi-controlled $Z$ gate sandwiched between $X$ gates on the qubits that should be $0$ in the target state. For $|1010\rangle$, qubits 0 and 2 should be $0$, so we apply $X$ gates to those before and after the MCZ.


In [ ]:
def grover_oracle_1010(qc):
    """Oracle marking |1010>. Qubit ordering: qc.qubits[0] is the rightmost bit."""
    # Apply X to qubits that should be 0 in |1010> (qubits 0 and 2)
    qc.x(0)
    qc.x(2)
    # Multi-controlled Z on 4 qubits
    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)
    # Undo the X gates
    qc.x(0)
    qc.x(2)
    return qc


def grover_diffusion(qc, n):
    """Standard diffusion operator (reflection around |+>^n)."""
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc


def build_grover_circuit(n=4, iterations=3):
    """Build a Grover circuit that searches for |1010> in n qubits."""
    qc = QuantumCircuit(n, n)
    # Uniform superposition
    qc.h(range(n))
    # Grover iterations
    for _ in range(iterations):
        grover_oracle_1010(qc)
        grover_diffusion(qc, n)
    # Measurement
    qc.measure(range(n), range(n))
    return qc


qc_test = build_grover_circuit(n=4, iterations=3)
qc_test.draw('mpl', fold=100)

Even at 4 qubits and 3 iterations the circuit is not small. Multi-controlled operations and Hadamard sandwiches accumulate quickly. This is why hardware matters here, since every extra gate is another chance for noise to eat your signal.

## Sanity check on the ideal simulator

Before we spend credits on real hardware, let's confirm the circuit does what we expect. We'll sweep the number of Grover iterations from 0 to 5 and plot the measured success probability. This should match the theoretical curve.


In [ ]:
def run_on_simulator(iterations, shots=4096):
    """Run Grover with `iterations` on the ideal simulator, return P(measure |1010>)."""
    qc = build_grover_circuit(n=4, iterations=iterations)
    sim = AerSimulator()
    result = sim.run(qc, shots=shots).result()
    counts = result.get_counts()
    return counts.get('1010', 0) / shots


iterations_range = range(0, 6)
sim_probs = [run_on_simulator(k) for k in iterations_range]

# Theoretical curve
theta = np.arcsin(1 / np.sqrt(16))
theory_probs = [np.sin((2 * k + 1) * theta) ** 2 for k in iterations_range]

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iterations_range, theory_probs, 'k--', label='Theoretical', linewidth=2, alpha=0.6)
ax.plot(iterations_range, sim_probs, 'o-', color='#a02580', label='Aer simulator', markersize=10, linewidth=2)
ax.set_xlabel('Grover iterations')
ax.set_ylabel('P(measure |1010>)')
ax.set_title('Grover success probability on the ideal simulator')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Peak success at k=3: {sim_probs[3]:.3f} (theory: {theory_probs[3]:.3f})")

The simulator matches theory almost exactly. Any tiny discrepancy is shot noise from a finite sample of 4,096 measurements per data point. This curve is the baseline. Anything we measure on real hardware that departs from it is a hardware effect, not an algorithm error.

## Enter the real world: three different QPUs

The qBraid platform gives you access to hardware from multiple vendors through one API. We'll use three devices that span the two main modalities available today:

| Backend | Vendor | Modality | Native two-qubit gate | Topology |
|---|---|---|---|---|
| `rigetti_ankaa_3` | Rigetti | Superconducting | ISWAP (or CZ) | Square lattice |
| `iqm_garnet` | IQM | Superconducting | CZ | Star with 5 arms |
| `aqt_marmot` | AQT | Trapped ion | XX (Molmer-Sorensen) | All-to-all |

*Device names above are illustrative. Check `provider.get_devices()` for currently available devices in your account and substitute accordingly.*

The instructive difference is in the last two columns. Trapped-ion systems have all-to-all connectivity, so any qubit can interact with any other. Superconducting systems only allow interactions between physically adjacent qubits, so any two-qubit gate between distant qubits must be routed via SWAP operations. This routing is often the dominant source of extra circuit depth and therefore extra noise.


In [ ]:
# Connect to qBraid
provider = QbraidProvider()

# List available devices (for reference; comment this out once you know the IDs you want)
# devices = provider.get_devices()
# for d in devices:
#     print(f"{d.id}: status={d.status}, num_qubits={d.num_qubits}")

# For this notebook we target three specific devices.
# Instructor: update these IDs to match currently available hardware in your account.
BACKENDS = {
    'Rigetti Ankaa-3':  'rigetti_ankaa_3',
    'IQM Garnet':       'iqm_garnet',
    'AQT Marmot':       'aqt_marmot',
}

# Colors for plotting
COLORS = {
    'Rigetti Ankaa-3': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT Marmot':      '#2d7a4f',
}

print("Configured backends:", list(BACKENDS.keys()))

## Transpilation: what actually runs on the chip

Before submitting, let's look at what each vendor's compiler produces from the same logical circuit. To me this is the most instructive moment in the notebook. The same abstract algorithm becomes very different concrete gate sequences depending on the target hardware.


In [ ]:
# Build the circuit we'll actually run
qc_grover = build_grover_circuit(n=4, iterations=3)

# Get device handles
devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}

# Transpile for each backend and record stats
transpile_stats = []
for name, device in devices.items():
    # Transpile using each backend's native gate set and topology.
    # qBraid handles Qiskit -> vendor-specific IR conversion under the hood.
    transpiled = transpile(qc_grover, backend=device.profile.get('qiskit_backend', None), optimization_level=2)
    stats = {
        'Backend': name,
        'Depth': transpiled.depth(),
        'Total gates': sum(transpiled.count_ops().values()),
        '2Q gates': sum(v for k, v in transpiled.count_ops().items() if k in ['cx', 'cz', 'iswap', 'ecr', 'rzx', 'xx']),
        'SWAPs inserted': transpiled.count_ops().get('swap', 0),
    }
    transpile_stats.append(stats)

# Pretty-print as a table
df = pd.DataFrame(transpile_stats).set_index('Backend')
df

Look at the differences carefully. The "2Q gates" column is usually the biggest driver of noise on today's hardware. Two-qubit gate fidelities are typically 99% at best, compared to 99.9% or better for single-qubit gates. If one backend needs twice as many two-qubit gates as another, you can predict which one will perform worse before you run it.

The "SWAPs inserted" column tells you how much the compiler had to work around limited connectivity. AQT (all-to-all) should show zero SWAPs. Rigetti and IQM will show some, and the exact number depends on which physical qubits the compiler chose to map your logical qubits onto.


## Submit to hardware

We'll run each backend at iteration counts $k \in \{1, 2, 3, 4\}$ so we can trace out the Grover curve on each device and compare it to the ideal curve. That's 4 circuits × 3 backends = 12 hardware jobs, each with 150 shots.

**Note on queue times.** Hardware jobs are queued, not instantaneous. Depending on device load this cell may take anywhere from a few minutes to a few hours to complete. The `job.wait_for_final_state()` calls block until each job finishes.


In [ ]:
SHOTS = 150
ITERATIONS_TO_TEST = [1, 2, 3, 4]

hardware_results = {name: {} for name in BACKENDS}
jobs = {name: {} for name in BACKENDS}

# Submit all jobs
for name, device in devices.items():
    for k in ITERATIONS_TO_TEST:
        qc = build_grover_circuit(n=4, iterations=k)
        job = device.run(qc, shots=SHOTS)
        jobs[name][k] = job
        print(f"Submitted: {name}, k={k}, job_id={job.id}")

# Wait for all jobs, collect results
for name in BACKENDS:
    for k in ITERATIONS_TO_TEST:
        job = jobs[name][k]
        result = job.result()
        counts = result.data.get_counts()
        # Normalize measurement bitstring format across vendors
        # (some return little-endian, some big-endian; qBraid provides a normalized interface)
        p_marked = counts.get('1010', 0) / SHOTS
        hardware_results[name][k] = p_marked
        print(f"{name}, k={k}: P(|1010>) = {p_marked:.3f}")

## Compare results

Now the payoff. Plot all three backends against the ideal curve on the same axes.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Theoretical curve (fine grid)
k_fine = np.linspace(0, 5, 100)
theory_fine = np.sin((2 * k_fine + 1) * theta) ** 2
ax.plot(k_fine, theory_fine, 'k--', label='Ideal (theory)', linewidth=2, alpha=0.5)

# Simulator points (from earlier)
ax.plot(list(iterations_range), sim_probs, 'k.', markersize=8, alpha=0.6, label='Aer simulator')

# Hardware points
for name in BACKENDS:
    ks = list(hardware_results[name].keys())
    ps = [hardware_results[name][k] for k in ks]
    ax.plot(ks, ps, 'o-', color=COLORS[name], label=name,
            markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Grover iterations', fontsize=12)
ax.set_ylabel('P(measure |1010>)', fontsize=12)
ax.set_title('Grover on real hardware: three different QPUs vs ideal curve', fontsize=13, pad=15)
ax.set_ylim(0, 1.05)
ax.set_xticks(range(0, 6))
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

The simulator curve should track the theoretical prediction almost exactly. The hardware curves are all *below* the ideal, and they may not even peak at $k=3$. Noise may instead cause the success probability to saturate or even decrease as you add more iterations, because each additional iteration adds more noisy gates that degrade the state.

So on today's hardware, more Grover iterations does not mean better answers past a certain point. There is an optimal number of iterations *for your specific hardware*, and it may be less than the theoretical optimum. No ideal-simulator experiment can teach you that.

In [ ]:
# Detailed comparison table
comparison_data = []
for name in BACKENDS:
    row = {'Backend': name}
    for k in ITERATIONS_TO_TEST:
        row[f'k={k}'] = f"{hardware_results[name][k]:.3f}"
    row['Ideal (k=3)'] = f"{theory_probs[3]:.3f}"
    row['Fidelity ratio (k=3)'] = f"{hardware_results[name][3] / theory_probs[3]:.3f}"
    comparison_data.append(row)

pd.DataFrame(comparison_data).set_index('Backend')

## What this tells us

The three backends will typically produce noticeably different results, and the differences correlate with two things:

1. **Two-qubit gate count in the transpiled circuit.** More two-qubit gates means more accumulated error. Trapped-ion systems typically have longer gate times but higher fidelity, which trades off against superconducting systems' faster but noisier gates.
2. **Native gate set and connectivity.** A backend whose native two-qubit gate is a poor match for the circuit's structure needs more decomposition, which means more gates. And a backend with limited connectivity needs SWAP operations to route qubits, which means even more gates.

The fidelity ratio in the table above (measured $P$ at $k=3$ divided by ideal $P$ at $k=3$) is a rough estimate of the overall circuit fidelity on each backend. If Rigetti gets ratio 0.4, IQM gets 0.6, and AQT gets 0.8, that is a concrete statement. For this specific circuit at this specific depth, AQT preserves twice as much of the ideal signal as Rigetti does.

But this ratio is not fixed. It depends on the algorithm, the number of qubits, and the specific gate patterns. A different algorithm might rank the backends differently.


## Design considerations for real hardware

Once you understand the differences, you can design better. A few practical patterns:

1. **Choose the modality that matches your algorithm structure.** Algorithms with high connectivity requirements (like some VQE ansatzes and QAOA on dense graphs) tend to run better on trapped-ion systems. Algorithms with local structure (like nearest-neighbor Ising simulation) can run efficiently on superconducting systems.

2. **Prefer shallower circuits when the mathematics allows.** For Grover, this means using fewer iterations than theory would suggest. For other algorithms, it means preferring approximate methods (e.g., low-depth variational circuits) over exact ones (e.g., deep QPE).

3. **Match the compiler's job.** Some compilers do heavy optimization; some do minimal. Passing `optimization_level=3` in Qiskit (or the equivalent in other frameworks) can cut gate counts, especially for high-connectivity circuits on limited-topology hardware.

4. **Benchmark before you commit.** Running a small version of your algorithm on all available hardware and looking at fidelity ratios is a cheap way to pick your target before scaling up. This notebook is essentially that benchmarking exercise for Grover.


## Where to go next

If your course covers algorithms beyond Grover, this pattern generalizes. Any circuit can be run through the same compare-across-backends workflow.

Recommended follow-ups:

- **Quantum Phase Estimation on three QPUs.** The companion notebook in this series (`Notebook 2`) does exactly this and adds zero-noise extrapolation as a mitigation strategy.
- **Change the marked state.** Rerun this notebook marking different bitstrings. Do the success rates change? Which bitstrings are hardest and why? (Hint: symmetry and native-gate compatibility.)
- **Explore mitigation.** Run Grover at $k=3$ with and without dynamical decoupling, or with measurement error mitigation. How much of the ideal fidelity can you recover?


---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. What would you want the next notebook in this series to cover?

Please submit your responses via the QUEST portal or reply to your instructor.
